In [0]:
%pip install                                                         \
        numpy                 sentencepiece       protobuf           \
        adlfs                 fsspec              azure-storage-blob \
        readability-lxml      w3lib               httpx              \
        beautifulsoup4        azure-ai-inference  packaging          \
        python-dotenv         openai              polars             \
        sentence-transformers tiktoken            fastembed

%restart_python

In [0]:
import os
import httpx
import numpy  as np
import scipy  as sp
from dotenv   import load_dotenv
from logging  import Logger, getLogger
from openai   import OpenAI, AsyncOpenAI

import adlfs
import polars as pl
from sklearn.decomposition import PCA, TruncatedSVD
import matplotlib.pyplot as plt

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import pyspark.sql.functions as F
from delta       import configure_spark_with_delta_pip

from azure.ai.inference.models import SystemMessage, UserMessage
from sentence_transformers     import SentenceTransformer

from llm_agent   import LlmAgent

In [ ]:
builder = SparkSession.builder\
            .config("spark.sql.sources.commitProtocolClass", "org.apache.spark.sql.execution.datasources.SQLHadoopMapReduceCommitProtocol")\
            .config("spark.sql.parquet.output.committer.class", "org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter")\
            .config("spark.mapreduce.fileoutputcommitter.marksuccessfuljobs","false")\
            .config("spark.sql.adaptive.enabled", True)\
            .config("spark.sql.shuffle.partitions", "auto")\
            .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "100MB")\
            .config("spark.sql.adaptive.coalescePartitions.enabled", True)\
            .config("spark.sql.dynamicPartitionPruning.enabled", True)\
            .config("spark.sql.autoBroadcastJoinThreshold", "10MB")\
            .config("spark.sql.session.timeZone", "Asia/Tokyo")\
            .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")\
            .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")\
            .config("spark.databricks.delta.write.isolationLevel", "SnapshotIsolation")\
            .config("spark.databricks.delta.optimizeWrite.enabled", True)\
            .config("spark.databricks.delta.autoCompact.enabled", True)
            # Delta Lake 用の SQL コミットプロトコルを指定
            # Parquet 出力時のコミッタークラスを指定
            # Azure Blob Storage (ABFS) 用のコミッターファクトリを指定
            # '_SUCCESS'で始まるファイルを書き込まないように設定
            # AQE(Adaptive Query Execution)の有効化
            # パーティション数を自動で調整するように設定
            # シャッフル後の1パーティションあたりの最小サイズを指定
            # AQEのパーティション合成の有効化
            # 動的パーティションプルーニングの有効化
            # 小さいテーブルのブロードキャスト結合への自動変換をするための閾値調整
            # SparkSessionのタイムゾーンを日本標準時刻に設定
            # Delta Lake固有のSQL構文や解析機能を拡張モジュールとして有効化
            # SparkカタログをDeltaLakeカタログへ変更
            # Delta Lake書き込み時のアイソレーションレベルを「スナップショット分離」に設定
            # 書き込み時にデータシャッフルを行い、大きなファイルを生成する機能の有効化
            # 書き込み後に小さなファイルを自動で統合する機能の有効化


spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [ ]:
# 例
# BRAND_NAME_FILE              = '01_&フェイス (&Face)'
# ADID_TOP_N_PERCENT_THRESHOLD = 0.2

# 手入力
BRAND_NAME_FILE              = ''
ADID_TOP_N_PERCENT_THRESHOLD = ''


In [0]:
# .env ファイルを読み込む
load_dotenv()

# 環境変数の取得
AI_FOUNDRY_ENDPOINT       = os.environ.get("AI_FOUNDRY_ENDPOINT")
AI_FOUNDRY_API_KEY        = os.environ.get("AI_FOUNDRY_API_KEY")
AI_FOUNDRY_MODEL          = os.environ.get("AI_FOUNDRY_MODEL")
LLM_MAX_TOKENS            = os.environ.get("LLM_MAX_TOKENS")
LLM_TEMPERATURE           = os.environ.get("LLM_TEMPERATURE")
LLM_TOP_P                 = os.environ.get("LLM_TOP_P")


# メモ：
# 環境変数・ウィジット経由でのパラメータの取得方法では、無条件にパラメータの型がStringに変換されてしまう
# そのため、受け取ったパラメータを適切に型変換する必要がある
LLM_MAX_TOKENS       = int(LLM_MAX_TOKENS)
LLM_TEMPERATURE      = float(LLM_TEMPERATURE)
LLM_TOP_P            = float(LLM_TOP_P)

# 簡易デバッグ用
# print(f'AI_FOUNDRY_ENDPOINT:       {AI_FOUNDRY_ENDPOINT}')       # セキュリティリスクのため、コメントアウト
# print(f'AI_FOUNDRY_API_KEY:        {AI_FOUNDRY_API_KEY}')        # セキュリティリスクのため、コメントアウト
print(f'AI_FOUNDRY_MODEL:          {AI_FOUNDRY_MODEL}')
print(f'LLM_MAX_TOKENS:            {LLM_MAX_TOKENS}')
print(f'LLM_TEMPERATURE:           {LLM_TEMPERATURE}')
print(f'LLM_TOP_P:                 {LLM_TOP_P}')

In [0]:
limits      = httpx.Limits(max_keepalive_connections=20, max_connections=300)
timeout     = httpx.Timeout(300.0, connect=5.0)
http_client = httpx.AsyncClient(limits=limits, timeout=timeout)
openai_cli  = AsyncOpenAI(
                    base_url=AI_FOUNDRY_ENDPOINT,
                    api_key=AI_FOUNDRY_API_KEY,
                    http_client=http_client,
                    max_retries=3
                )
llmClient   = LlmAgent(openai_cli, AI_FOUNDRY_MODEL, int(LLM_MAX_TOKENS), float(LLM_TEMPERATURE), float(LLM_TOP_P))

with open(f'ブランドLP/{BRAND_NAME_FILE}', 'r', encoding='utf-8') as f:
    res_str = f.read()
print(res_str)

### 前処理データ部

INPUT:
- LLMによる商品LPの分析結果
- ADIDの行動特徴テーブル

OUTPUT:
- [1, コホートキャプション数]なLPのスポット係数ベクトル

In [0]:
messages  = []
messages.append(SystemMessage(content=(
				"あなたは提供された「商品分析結果」を基に、その商品の利用文脈（Micro-Context）を特定するマーケティングの専門家です。\n"
				"提供された商品情報を分析し、その商品が「最高に輝く具体的なシーン（適合）」と「全く役に立たない、あるいはミスマッチなシーン（不適合）」を洗い出してください。\n\n"
				
				"【重要な指示】\n"
                "出力するキーワードは、ベクトル検索のクエリとして使用されます。\n"
				"そのため、単一の一般名詞（例：「公園」「オフィス」）は **禁止** です。\n"
                "必ず **「場所＋状況」** または **「属性＋場所」** の複合キーワード（Micro-Context）を選定してください。\n\n"
    			"抽出する単語は、単なる一般名詞ではなく、「誰が・どこで・何をしているか」がありありと想像できるような、具体的かつ解像度の高いキーワードを選定してください。\n"
    			"特に「場所」に関しては、大分類（例：公園）ではなく、詳細な施設タイプ（例：ドッグラン、親水広場）や、利用目的が明確なスポット名を優先してください。\n\n"
                "LPに直接記載がなくても、商品の特性から論理的に推測されるシーンは積極的に広げて記述してください。\n\n"
                "・NG例: 「ジム」「キャンプ」「サラリーマン」\n"
                "・OK例: 「深夜の24時間ジム」「雨上がりのオートキャンプ場」「満員電車の通勤客」「コンセントのあるカフェ席」\n\n"

				"【出力形式（厳守）】\n"
				"回答は必ず以下のJSON形式のみを出力してください。\n"
				"各単語をキーとし、その関連度の強さ（重み）を 0.0〜1.0 の数値で値として設定してください。\n"
				"Markdown記法（```json 等）は含めず、生のJSONテキストのみを返してください。\n\n"
                "・Positiveとして、確信度の高い上位10個程度を抽出してください。\n\n"
                "・Negativeとして、確信度の高い上位7個程度を抽出してください。\n\n"
					
				"""
				{
					"positive": {"複合キーワードA": 0.89, "キーワードB": 0.70, ...},
					"negative": {"複合キーワードC": 0.91, ...},
				}
				"""
				"\n\n"
				
				"【分析の視点】\n"
				"--- positive（適合）：商品が必須となる、または魅力を最大化する文脈 ---\n"
				"1. 具体的な施設・スポット（Places）：\n"
				"   - 抽象的な「店」「屋外」はNG。\n"
				"   - 「24時間ジム」「オートキャンプ場」「コワーキングスペース」など、行動が特定できる施設名。\n"
				"2. 利用シーン・瞬間（Scenes）：\n"
				"   - 「通勤ラッシュ」「運動後のシャワー」「子供の寝かしつけ」など、具体的なタイムラインや状況。\n"
				"3. ターゲットの属性・状態（Traits）：\n"
				"   - 「健康志向」のような広い言葉より、「糖質制限中」「リモートワーク疲れ」など具体的な状態。\n\n"
                "4. 物理的適合 (Physical Fit):\n"
                "   - 商品のサイズ、電源、耐久性が、その場所の設備・環境と完璧に噛み合うか。\n"
                "   - マグネットでくっつく防水Bluetoothスピーカー  →  「ユニットバスの壁面」「雨の日のキャンプのタープ下」"
				"5. 心理的・行動的適合 (Contextual Fit):\n"
                "   - その場所にいる人の「特定の悩み」を解決するか。\n"
				"   - 周囲の音を消すデジタル耳栓  →  「いびきが気になるカプセルホテル」「瞑想に集中したいヨガスタジオの隅」"
				
				"--- negative（不適合）：商品の機能が死ぬ、または邪魔になる文脈 ---\n"
				"1. 阻害要因となる場所（Places）：\n"
				"   - 商品のスペック（大きさ、音、電源有無など）的に使えない場所（例：図書館、満員電車）。\n"
				"2. 無意味なシーン（Scenes）：\n"
				"   - その商品をあえて使う必要がない状況。\n\n"
                "3. 環境不適合 (Environmental Mismatch):\n"
                "   - 商品を使うには狭すぎる、暗すぎる、うるさすぎる、静かすぎる場所。\n"
                "4. マナー・ルール違反 (Social Mismatch):\n"
                "   - その場所でその商品を使うことが「白い目」で見られる、あるいは禁止されている。\n"
				
				"【重み（スコア）の基準】\n"
				"・1.0に近いほど：その傾向が非常に強い、確信度が高い\n"
				"・0.0に近いほど：関連性が薄い\n"
                "・1.0: 完全にその商品の独壇場である（または絶対に使用不可である）。\n"
                "・0.8: 非常に相性が良い（または強い懸念がある）。\n"
                "・0.5: 条件による（今回は出力対象外）。\n"
				"・positiveの場合：適合度の高さ\n"
				"・negativeの場合：不適合度の高さ（明確に避けるべき度合い）"
			)))
messages.append(UserMessage(content=res_str))
analysis_data = await llmClient.complete(messages)
analysis_data

In [ ]:
# 特定の地点に対するADIDリストを取得
POLYGONLIST_TABLE = "adinte_adintedmp.envprod.polygonlist"
sdf_polygonlist   = spark.read.table(POLYGONLIST_TABLE)\
							.select(['project', 'caption', 'extraction_id', 'branch_id', 'spot_id'])\
                            .filter(col('extraction_id') == 44082)
ADIDLIST_TABLE    = "adinte_adintedmp.envprod.source"
sdf_adidlist      = spark.read.table(ADIDLIST_TABLE)\
							.select(['extraction_id', 'branch_id', 'spot_id', 'adid'])\
                            .filter(col('extraction_id') == 44082)\
							.join(sdf_polygonlist, on=['extraction_id', 'branch_id', 'spot_id'], how='inner')\
                            .select(['caption', 'adid'])\
                            .dropDuplicates()

target_adidlist = [row['adid'] for row in sdf_adidlist.select('adid').dropDuplicates().collect()]
pdf_adidlist    = sdf_adidlist.toPandas()
pdf_adidlist

In [0]:
# メモ：
# cohort.npzは以下のようなデータ構成になっている
# cohort.npz
#     |-- data              : 計算済みコホート係数行列
#     |-- indices           : データの位置指定子(列)
#     |-- indptr            : 行ごとのスライスインデックス
#     |-- shape             : コホート係数行列の形(ADIDのリスト × コホートキャプションのリスト)
#     |-- adid_list         : ADIDのリスト(列)
#     |-- business_codelist : コホートキャプションIDのリスト(行)
TARGET      = "/Volumes/stgadintedmpadintedi/featurestore/behaviorvector/cohort.npz"
npz         = np.load(TARGET, allow_pickle=True)
np_cohort   = sp.sparse.csr_matrix((npz["data"], npz["indices"], npz["indptr"]), shape=tuple(npz["shape"]))
np_adidlist = npz["adid_list"]
np_codelist = npz["business_codelist"]

# 特定のADIDのみの抜き出し
indexer     = pd.Index(np_adidlist)
indices     = indexer.get_indexer(target_adidlist)
indices     = indices[indices != -1]
np_cohort   = np_cohort[indices, :]
np_adidlist = np_adidlist[indices]


# メモ：
# L2正規化を行うにあたって、疎行列専用に書く必要がある
# 密行列であれば簡単に書くことが可能であるが、今回の要件に適合しない
# そのため疎行列な対角行列を経由することにより、これを実現することとした
# 
# 当初すでにL2正規化は適用済みであったが、時々適用されていない状態になることがあるようだ
# そのため、改めて適用することとした
# L2正規化
if not np.allclose(np_cohort.power(2).sum(axis=1), 1.0):
	l2norm_vector = sp.sparse.linalg.norm(np_cohort, axis=1)
	np_cohort     = sp.sparse.diags(1 / np.maximum(l2norm_vector, 1e-10)) @ np_cohort


# CODELISTに対応するキャプションの文脈行列を取得
CAPTION_CONTEXT_MATRIX = "cohort_caption_matrix.npz"
npz                    = np.load(CAPTION_CONTEXT_MATRIX, allow_pickle=True)
spots_matrix           = npz["data"]
relational_codelist    = npz["business_codelist"]
relational_spots       = npz["business_placelist"]
dict_code2name         = npz["dict_code2name"].item()

# 簡易チェック
assert np.allclose(np_cohort.power(2).sum(axis=1), 1.0), "Not all row vectors have a unit length (squared norm != 1.0)."
assert np.array_equal(relational_codelist, np_codelist), "There are unregistered keys."
assert np.array_equal(np.array([[code, *dict_code2name[code]] for code in np_codelist]), relational_spots), "Location list is inconsistent."
assert spots_matrix.shape[0] == np_cohort.shape[1], "Dimension mismatch: spots_matrix rows (count: {}) do not align with np_cohort columns (count: {})" .format(spots_matrix.shape[0], np_cohort.shape[1])

print(np_cohort.shape)
print(spots_matrix.shape)

In [0]:
items_positive   = list(analysis_data['positive'].items())
items_negative   = list(analysis_data['negative'].items())
lp_keywords      = [key for key, val in items_positive] + [ key for key, val in items_negative]
lp_weights       = [val for key, val in items_positive] + [-val for key, val in items_negative]

total_weight     = np.sum(np.abs(lp_weights))
lp_weights       = np.array(lp_weights) / total_weight

# model            = SentenceTransformer('cl-nagoya/ruri-v3-130m')
model            = SentenceTransformer('BAAI/bge-m3')
lp_vector        = lp_weights.reshape(1, -1)                             # 1 × キーワード数M
lp_matrix        = model.encode(lp_keywords, normalize_embeddings=True)  # キーワード数M × 1024
lp_coefficient   = lp_vector @ lp_matrix @ spots_matrix.T                # LPのスポット係数

# L2正則化
l2norm_vector    = np.linalg.norm(lp_coefficient, axis=1, keepdims=True)
lp_coefficient   = lp_coefficient / np.maximum(l2norm_vector, 1e-10)

lp_coefficient

### ADID毎のスコア・理由を算出

In [0]:
# ADID毎のスコアを算出
np_scored         = (np_cohort @ lp_coefficient.T).flatten()
indices           = np.argsort(np_scored)[::-1]
sorted_adidlist   = np_adidlist[indices]
sorted_scored     = np_scored[indices]

# メモ：
# 閾値について
# 相関係数0.7程度と指定を受けたが、高次元空間において、この値はあり得ない
# そのため2次元における相関係数 0.7 ≒ 0.70710678 ≒ 1 / √2 であると仮定して
# 1 / √851 ≒ 0.03427956 ≒ 0.034

# 閾値の設定(上位N%地点を指定する)(スコアが正であることを条件とする)
REASON_THRESHOLD = max(0, np.quantile(sorted_scored, 1 - ADID_TOP_N_PERCENT_THRESHOLD))

# フレームワークへの変換
pldf_scores       = pl.LazyFrame({
    						'ADID'  : sorted_adidlist, 
                            'score' : sorted_scored, 
                        })\
						.cast({
                            'ADID'  : pl.String,
                            'score' : pl.Float32,
                        })\
                        .select(['ADID', 'score'])

pldf_adidlist      = pl.from_pandas(pdf_adidlist).lazy()\
						.rename({'adid':'ADID'})\
						.join(pldf_scores, on='ADID', how='inner')\
                        .select(['caption', 'ADID', 'score'])\
                        .filter(pl.col('score') > REASON_THRESHOLD)\
                        .collect()

pldf_store_sum     = pldf_adidlist\
						.group_by(pl.col('caption'), maintain_order=True)\
                        .agg(pl.count().alias('total_count'))\
                        .select(['caption', 'total_count'])
count              = pldf_adidlist.select(pl.col("ADID").n_unique()).item()

print(f"処理対象: {BRAND_NAME_FILE}")
print(f"ユニークなADID数: {count}")
print(pldf_adidlist)

In [0]:
# 可視化(ヒストグラム)
TARGET = f"/Volumes/stgadintedmpadinteds/envdev/sandbox/Cohort_AI_Agent_Siseido/{BRAND_NAME_FILE}_visible_histogram.png"
quantiles = np.quantile(sorted_scored, np.arange(0.1, 1.0, 0.1))
plt.figure(figsize=(10, 6))

plt.hist(sorted_scored, bins=30, color='skyblue', edgecolor='black')
for i, q in enumerate(quantiles):
    plt.axvline(q, color='red', linestyle='--', linewidth=1, alpha=0.6)
    # グラフ上部に「10%」「20%」などのラベルを表示（任意）
    plt.text(q, plt.ylim()[1] * 1.01, f'{(i+1)*10}%', 
             color='red', fontsize=9, ha='center', va='bottom')

plt.xlabel('Score')
plt.ylabel('Frequency')
plt.title('Histogram of Scored Data')
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig(TARGET, dpi=300, bbox_inches='tight')
plt.show()


# 可視化(散布図)
# TARGET = f"/Volumes/stgadintedmpadinteds/envdev/sandbox/Cohort_AI_Agent_Siseido/{BRAND_NAME_FILE}_visible_PCA.png"
# pca = TruncatedSVD(n_components=2)
# vectors_pca = pca.fit_transform(np_cohort)
# plt.figure(figsize=(10, 6))
# plt.scatter(vectors_pca[:, 0], vectors_pca[:, 1], label='ADID', alpha=0.3, s=1)
# plt.title('PCA Visualization')
# plt.grid(True, linestyle='--', alpha=0.7)
# plt.tight_layout()
# plt.savefig(TARGET, dpi=300, bbox_inches='tight')
# plt.show()


TARGET = f"/Volumes/stgadintedmpadinteds/envdev/sandbox/Cohort_AI_Agent_Siseido/{BRAND_NAME_FILE}_all.parquet"
pldf_adidlist.write_parquet(TARGET, compression="snappy")
TARGET = f"/Volumes/stgadintedmpadinteds/envdev/sandbox/Cohort_AI_Agent_Siseido/{BRAND_NAME_FILE}_separate.parquet"
pldf_store_sum.write_parquet(TARGET, compression="snappy")